# Week 09 - Object Detection I: R-CNN to Faster R-CNN

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- State the object detection problem: **localisation + classification**.
- Compute and interpret **IoU** and apply **non-maximum suppression (NMS)**.
- Explain the evolution **R-CNN → Fast R-CNN → Faster R-CNN**.
- Run a pretrained **Faster R-CNN** and interpret boxes, labels and scores.
- Explain **precision, recall and mAP** and why detection is not just accuracy.

### The detection problem
Classification answers *what*. Detection answers *what* **and** *where*: it outputs a set of **bounding boxes**, each with a class label and a confidence score.

## 1. Setup
> This notebook downloads pretrained weights. A GPU runtime in Colab is recommended (*Runtime > Change runtime type > T4 GPU*).

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install torch torchvision matplotlib opencv-python ipywidgets

In [ ]:
import sys
sys.path.append("resources/scripts")
import cv2, numpy as np, torch
from PIL import Image
from cvhelpers import show, concept_map
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. The R-CNN family at a glance

In [ ]:
concept_map([
    "R-CNN (2014): ~2000 selective-search regions, each through a CNN. Accurate but very slow.",
    "Fast R-CNN (2015): one CNN pass over the image + ROI pooling. Much faster.",
    "Faster R-CNN (2015): adds a Region Proposal Network (RPN) - end-to-end.",
    "Modern two-stage detectors are accurate but still heavier than one-stage (Week 10)."
], title="Evolution of two-stage detectors")

## 3. Guided example - Intersection over Union (IoU)
$$\mathrm{IoU} = \frac{\text{area of overlap}}{\text{area of union}}$$
IoU is used to decide whether a prediction is a true positive (usually IoU >= 0.5).

In [ ]:
def iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

gt = [100, 100, 300, 300]
for shift in [0, 30, 80, 150]:
    pred = [100 + shift, 100, 300 + shift, 300]
    print(f"shift={shift:3d}px  IoU={iou(gt, pred):.3f}")

###  Interactive exploration - IoU
Shift and resize the prediction box and watch IoU. Notice how quickly IoU falls as the box drifts.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.pyplot as plt

def iou_demo(dx=20, dy=0, scale=100):
    pred = [100 + dx, 100 + dy, 100 + dx + 200 * scale // 100, 100 + dy + 200 * scale // 100]
    canvas = np.ones((420, 520, 3), np.uint8) * 255
    cv2.rectangle(canvas, (gt[0], gt[1]), (gt[2], gt[3]), (0, 160, 0), 3)
    cv2.rectangle(canvas, (pred[0], pred[1]), (pred[2], pred[3]), (0, 0, 255), 3)
    cv2.putText(canvas, f"IoU = {iou(gt, pred):.3f}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
    show(canvas, titles=["green = ground truth, red = prediction"], figsize=(7, 6))

interact(iou_demo,
         dx=widgets.IntSlider(min=-80, max=80, step=5, value=20),
         dy=widgets.IntSlider(min=-80, max=80, step=5, value=0),
         scale=widgets.IntSlider(min=60, max=140, step=5, value=100))

## 4. Guided example - non-maximum suppression (NMS)
A detector produces many overlapping boxes for the same object. NMS keeps the highest-scoring box and removes others with IoU above a threshold.

In [ ]:
boxes = np.array([[100, 100, 300, 300],
                  [110, 105, 305, 295],   # almost the same object
                  [400, 120, 560, 300],
                  [405, 125, 555, 295]],  # another object
                  dtype=np.float32)
scores = np.array([0.95, 0.80, 0.88, 0.70], dtype=np.float32)
keep = cv2.dnn.NMSBoxes(boxes.tolist(), scores.tolist(), score_threshold=0.3, nms_threshold=0.5)
print("Kept indices after NMS:", np.array(keep).ravel().tolist())
print("Kept scores:", scores[np.array(keep).ravel()])

## 5. Guided example - pretrained Faster R-CNN inference
We load a model trained on **COCO** (80 classes) and run it on a test image.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights).eval()
categories = weights.meta["categories"]
print("Number of categories (incl. background):", len(categories))

In [ ]:
img_bgr = cv2.imread("resources/images/test_image.jpeg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
tensor = weights.transforms()(Image.fromarray(img_rgb)).unsqueeze(0)

with torch.no_grad():
    prediction = model(tensor)[0]

print("Detected boxes:", len(prediction["boxes"]))
for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
    if score > 0.7:
        print(f"{categories[label]:15s} score={score:.2f} box={[int(v) for v in box]}")

###  Interactive exploration - confidence threshold
Lower thresholds reveal more (but less reliable) detections. This is the precision-recall trade-off in action.

In [ ]:
def detect_demo(conf=0.7):
    vis = img_bgr.copy()
    count = 0
    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        if score < conf:
            continue
        count += 1
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, f"{categories[label]} {score:.2f}", (x1, y1 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    show(vis, titles=[f"conf >= {conf:.2f}  ->  {count} detections"], figsize=(8, 6))

interact(detect_demo, conf=widgets.FloatSlider(min=0.1, max=0.95, step=0.05, value=0.7))

## 6. Guided example - precision, recall and mAP
For a given IoU threshold and class:
$$\text{Precision}=\frac{TP}{TP+FP},\qquad \text{Recall}=\frac{TP}{TP+FN}$$
**Average Precision (AP)** is the area under the precision-recall curve; **mAP** averages AP over classes (and often over IoU thresholds 0.5:0.95).

In [ ]:
def precision_recall(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    return p, r

for tp, fp, fn in [(8, 2, 2), (8, 8, 2), (10, 0, 0), (5, 1, 5)]:
    p, r = precision_recall(tp, fp, fn)
    print(f"TP={tp} FP={fp} FN={fn}  ->  precision={p:.2f}  recall={r:.2f}")

### Annotation formats you will meet
- **Pascal VOC**: one XML per image with `<bndbox>` coordinates.
- **COCO**: one JSON with `images`, `annotations` and `categories`; boxes are `[x, y, width, height]`.
- **YOLO**: one `.txt` per image, one line per object: `class cx cy w h` (normalised 0-1).

Getting annotations right is usually harder than training the model.

## 7. Exercise (complete the code)

1. Write `best_match(pred_boxes, gt_box)` that returns the highest IoU between a ground-truth box and a list of predicted boxes, plus the index of that prediction.
2. Use it to label each prediction as **TP** (IoU >= 0.5) or **FP** for a single object.
3. Compute precision and recall for that object.

In [ ]:
# TODO: implement best_match and compute precision/recall


## 8. Challenge (independent)

Collect **three** of your own photos with multiple objects, run Faster R-CNN, and evaluate the detections by eye. For each image, record the false positives and the missed objects. Then propose two ways to improve the results (e.g. more training data, a different detector, or a confidence/NMS tuning).

In [ ]:
# Your code here


## 9. Check your understanding (Q&A)

<details><summary><b>Q1. Why can a very high-confidence detection still be wrong?</b></summary>

Confidence reflects the model's internal probability, not the true correctness. A model can be confidently wrong, especially on out-of-distribution or blurred inputs.
</details>

<details><summary><b>Q2. What problem does NMS solve?</b></summary>

Detectors output many overlapping boxes for the same object. NMS removes duplicates and keeps the best box.
</details>

<details><summary><b>Q3. Why is accuracy a poor metric for object detection?</b></summary>

Detection has a variable number of objects and must consider localisation quality. Precision, recall and mAP better reflect performance.
</details>

<details><summary><b>Q4. What is the key difference between R-CNN and Faster R-CNN?</b></summary>

R-CNN uses an external region proposal method and runs a CNN per region. Faster R-CNN learns proposals with a Region Proposal Network, making detection end-to-end and far faster.
</details>

## 10. Further reading & self-exploration
- torchvision object detection models: https://pytorch.org/vision/stable/models.html
- COCO dataset and evaluation: https://cocodataset.org/
- R-CNN paper: Girshick et al. (2014). Faster R-CNN paper: Ren et al. (2015).
- LearnOpenCV detection tutorials: https://learnopencv.com/
- Wikipedia - Object detection: https://en.wikipedia.org/wiki/Object_detection
- OpenCV DNN module: https://docs.opencv.org/4.x/d2/d58/tutorial_table_of_content_dnn.html

**Try next:** visualise the precision-recall curve for one COCO class using `torchmetrics`.

## 11. Key takeaways
- Detection = localisation + classification.
- **IoU** measures box overlap; **NMS** removes duplicates.
- Two-stage detectors (R-CNN family) are accurate but heavier.
- Evaluate with **precision, recall and mAP**, not accuracy.
- Annotation quality dominates real-world performance.